# 模型的调用
Langchain提供了几种核心的调用方式，主要是 invoke()、stream() 和 batch() 方法，以及异步版本 ainvoke()、astream()和 abatch()。 官方文档：https://langchain-doc.cn/v1/python/langchain/models.html#%E8%B0%83%E7%94%A8
- invoke(): 阻塞式，一次性返回完整结果问答、批处理任务、无需实时反馈的场景。
- stream()：流式输出，实时返回每个token聊天机器人、长文本生成、需要提升用户体验的交互应用。
- atch() ：批量处理多个输入高并发场景，需要同时处理大量请求。

In [ ]:
# invoke()
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
# 需要显示加载配置文件
load_dotenv()  # 默认读取当前目录的 .env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 初始化模型
model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

conversation = [
    {"role": "system", "content": "你是一个将中文翻译成英语的有用助手。"},
    {"role": "user", "content": "翻译：我喜欢编程。"},
]

response = model.invoke(conversation)
print(response)

content='I like programming.' additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer user. User asks Chinese translate to English: "我喜欢编程。" We need provide translation. Simple. Need maybe just "I like programming." Could also "I enjoy programming." Need respond. Ensure no extra? They asked 翻译. We can give English. Final.'} response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 46, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 57, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 46}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'c4837f56-0c0e-4815-92ce-3f0f8ec3b804', 'finish_reason':

In [ ]:
# 使用内置的消息类（如 SystemMessage, HumanMessage, AIMessage），将消息对象列表输入模型
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import os
import json
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
# 需要显示加载配置文件
load_dotenv()  # 默认读取当前目录的 .env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 初始化模型
model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

conversation = [
    # {"role": "system", "content": "你是一个将中文翻译成英语的有用助手。"},
    # {"role": "user", "content": "翻译：我喜欢编程。"},
    SystemMessage(content="你是一个python高级工程师"),
    HumanMessage(content="python协程是什么？")
]

response = model.invoke(conversation)
print(response)

TypeError: Object of type AIMessage is not JSON serializable

In [ ]:
response = model.invoke([HumanMessage(content="2的4次方为多少？")])
print(response.text)

2 的 4 次方是 **16**。


- invoke() 返回的是一个AiMessage对象，详细信息可以通过 rich 输出。
具体参数含义:

In [11]:
from rich import print as rprint
# 打印响应
print(type(response))
# <class 'langchain_core.messages.ai.AIMessage
rprint(response)

<class 'langchain_core.messages.ai.AIMessage'>


AIMessage(
    content='2 的 4 次方是 **16**。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '我们需要回答中文。问题：2的4次方为多少？ 2^4=16。需要简洁。可以直接回答16。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 42,
            'prompt_tokens': 38,
            'total_tokens': 80,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 28,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 0,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 38
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': 'bafa8f95-a490-4526-8337-1a56a1f58d51',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a101c2-a8da-7080-94b3-83192d375e0a-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 38,
        'output_tokens': 42,
        'total_tokens': 80,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 28}
    }
)

In [12]:
# 获取响应详情
response = model.invoke("用一句话解释什么是 AI")
# 1. 获取回复内容
print("AI 回复:", response.content)
# 2. 获取响应元数据
metadata = response.response_metadata
print(f"使用的模型: {metadata['model_name']}")
print(f"结束原因: {metadata['finish_reason']}")
print(f"模型提供商：{metadata['model_provider']}\n")
# 3. 获取 Token 使用情况
usage = metadata.get('token_usage', {})
print(f"输入 tokens: {usage.get('prompt_tokens')}")
print(f"输出 tokens: {usage.get('completion_tokens')}")
print(f"总计 tokens: {usage.get('total_tokens')}")
# 4. 获取消息 ID
print(f"消息 ID: {response.id}")


AI 回复: AI 是让机器模拟人类智能，从而能够感知、学习、推理、决策并完成各种任务的技术。
使用的模型: deepseek-flash
结束原因: stop
模型提供商：deepseek

输入 tokens: 35
输出 tokens: 134
总计 tokens: 169
消息 ID: lc_run--01a101d0-5615-71a3-8f3a-8ed0e24a1480-0


In [16]:
# stream() 流式输出
for chunk in model.stream("用一句话解释什么是 AI? 再详细一点"):
    print(chunk.text,end="",flush=True)



**一句话：** AI（人工智能）是让机器通过数据、算法和算力表现出感知、学习、推理、决策、生成等类似人类智能行为的技术与系统总称。

**再详细一点：**

1. **核心目标**  
   AI 不是让机器“变成人”，而是让机器能完成过去通常需要人类智能才能完成的任务，比如识别图像、理解语言、翻译、下棋、写文章、做推荐、开车等。

2. **怎么实现**  
   传统程序靠人写死规则：如果 A 就做 B。  
   AI 尤其是现代 AI，更多是靠**数据 + 算法 + 算力**，从大量样本中找规律，再用这些规律去预测、分类或生成新内容。比如给模型看过海量文本后，它学会“下一个词最可能是什么”。

3. **主要能力**  
   - **感知**：看懂图片、听懂语音、识别视频  
   - **理解与生成**：理解问题、写文章、画图、写代码  
   - **推理与决策**：分析信息、做判断、规划路径  
   - **学习与适应**：从数据中改进表现  
   - **交互**：用自然语言和人对话

4. **常见技术**  
   包括机器学习、深度学习、神经网络、强化学习、大语言模型、计算机视觉、自然语言处理等。ChatGPT、人脸识别、推荐系统、自动驾驶都属于 AI 的应用形态。

5. **分类**  
   - **窄 AI / 弱 AI**：只擅长特定任务，比如下棋、翻译、推荐视频。目前绝大多数 AI 都属于这一类。  
   - **通用 AI / 强 AI**：像人一样能跨领域学习、推理和迁移。目前还没有实现。  
   - **超级 AI**：全面超越人类的假设性智能，仍属未来设想。

6. **重要边界**  
   当前 AI 很强大，但通常没有意识、情感和真正的“理解”。它基于统计规律工作，可能出错、产生偏见或“幻觉”，也需要人类监督和规范。

所以，AI 本质上是一类**用计算系统模拟和扩展智能行为的工具**：它能帮人看、听、说、写、判断和决策，但它不是人，也不等于有意识的生命。

In [ ]:
# batch() 批量调用